# Step 2 - Chunk Scraped Pages

Builds `knowledge_base/chunks.json` from `knowledge_base/pages/*.txt`.

In [2]:
import json
import re
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name.lower() == "backend":
    ROOT = ROOT.parent

PAGES_DIR = ROOT / "knowledge_base/pages"
OUT_FILE = ROOT / "knowledge_base/chunks.json"
MIN_CHARS = 30

print(f"[SETUP] Input dir: {PAGES_DIR}")
print(f"[SETUP] Output file: {OUT_FILE}")
print(f"[SETUP] Min chars: {MIN_CHARS}")

[SETUP] Input dir: d:\Users\Danish\Desktop\Projects\Attendance-Web\knowledge_base\pages
[SETUP] Output file: d:\Users\Danish\Desktop\Projects\Attendance-Web\knowledge_base\chunks.json
[SETUP] Min chars: 30


In [6]:
def normalize_text(value: str) -> str:
    text = re.sub(r"\r\n?", "\n", value or "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_page_and_url(content: str, source_file: str) -> tuple[str, str]:
    page_name = source_file
    url = ""

    page_match = re.search(r"^PAGE:\s*(.+)$", content, re.MULTILINE)
    if page_match:
        page_name = normalize_text(page_match.group(1))

    url_match = re.search(r"^URL:\s*(.+)$", content, re.MULTILINE)
    if url_match:
        url = normalize_text(url_match.group(1))

    return page_name, url


def strip_page_headers(content: str) -> str:
    lines = content.splitlines()
    body_lines = [
        line
        for line in lines
        if not re.match(r"^\s*(PAGE|URL):", line, flags=re.IGNORECASE)
    ]
    return normalize_text("\n".join(body_lines))

In [3]:
def sectionize(content: str) -> list[tuple[int, str, str]]:
    pattern = re.compile(
        r"={10,}\s*\n\[\s*SECTION\s+(\d+)\s*:\s*(.*?)\s*\]\s*\n={10,}\s*\n(.*?)(?=\n={10,}\s*\n\[\s*SECTION\s+\d+\s*:|\Z)",
        re.DOTALL | re.IGNORECASE,
    )

    sections = []
    for match in pattern.finditer(content):
        section_num = int(match.group(1))
        heading = normalize_text(match.group(2))
        body = normalize_text(match.group(3))
        sections.append((section_num, heading, body))

    return sections


def split_by_delimiter(content: str) -> list[str]:
    return [normalize_text(part) for part in re.split(r"={3,}", content) if normalize_text(part)]

In [4]:
def build_chunks() -> list[dict]:
    files = sorted(PAGES_DIR.glob("*.txt"))
    if not files:
        raise FileNotFoundError(f"No .txt files found in: {PAGES_DIR}")

    chunks = []
    next_id = 0
    empty_files = []

    for file_path in files:
        raw = file_path.read_text(encoding="utf-8", errors="ignore")
        page_name, url = parse_page_and_url(raw, file_path.stem)
        body_text = strip_page_headers(raw)

        sections = sectionize(raw)
        if sections:
            for section_num, heading, body in sections:
                if len(body) < MIN_CHARS:
                    continue
                chunks.append(
                    {
                        "id": f"chunk_{next_id:04d}",
                        "page_name": page_name,
                        "url": url,
                        "source_file": file_path.name,
                        "section_num": section_num,
                        "section_heading": heading,
                        "content": body,
                    }
                )
                next_id += 1
            continue

        if len(body_text) < MIN_CHARS:
            empty_files.append(file_path.name)
            continue

        section_num = 0
        for part in split_by_delimiter(body_text):
            cleaned = re.sub(r"^\[\s*SECTION\s*\d+.*?\]\s*", "", part, flags=re.IGNORECASE)
            cleaned = normalize_text(cleaned)
            if len(cleaned) < MIN_CHARS:
                continue

            chunks.append(
                {
                    "id": f"chunk_{next_id:04d}",
                    "page_name": page_name,
                    "url": url,
                    "source_file": file_path.name,
                    "section_num": section_num,
                    "section_heading": "",
                    "content": cleaned,
                }
            )
            next_id += 1
            section_num += 1

    if empty_files:
        print(f"[CHUNK] Skipped empty/no-content files: {len(empty_files)}")

    return chunks

In [7]:
chunks = build_chunks()
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

source_files = sorted({item["source_file"] for item in chunks})
print(f"[CHUNK] Source files used: {len(source_files)}")
print(f"[CHUNK] Chunks generated: {len(chunks)}")
if chunks:
    print(f"[CHUNK] First chunk id: {chunks[0]['id']}")
    print(f"[CHUNK] Last chunk id: {chunks[-1]['id']}")
print("[CHUNK] Done")

[CHUNK] Skipped empty/no-content files: 25
[CHUNK] Source files used: 1
[CHUNK] Chunks generated: 25
[CHUNK] First chunk id: chunk_0000
[CHUNK] Last chunk id: chunk_0024
[CHUNK] Done
